In [1]:
# -*- coding: utf-8 -*-
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
# implied.
# See the License for the specific language governing permissions and
# limitations under the License.


# Imports

In [2]:
import pandas as pd
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.model_selection import GridSearchCV
import numpy as np
from sklearn.decomposition import TruncatedSVD
import os
import wandb

from util.preprocessing import TweetPreprocessor 
import joblib

# Consts

In [3]:
MODEL_DIR = os.path.join(os.getcwd(), 'models')
DATASETS_DIR = os.path.join(os.getcwd(), 'datasets')

# Dataset

In [4]:
train_df = pd.read_csv(os.path.join(DATASETS_DIR, "one_tweet_dataset_train.csv"))
test_df = pd.read_csv(os.path.join(DATASETS_DIR, "one_tweet_dataset_test.csv"))
val_df = pd.read_csv(os.path.join(DATASETS_DIR, "one_tweet_dataset_val.csv"))

In [5]:
x_train, y_train = train_df["text"], train_df["gender_label"]
x_test, y_test = test_df["text"], test_df["gender_label"]
x_val, y_val = val_df["text"], val_df["gender_label"]

# Initiate pipeline

In [6]:
pipeline = Pipeline([
    ("preprocessor", TweetPreprocessor()),
    ("features", FeatureUnion([
        ("tfidf_word", TfidfVectorizer(analyzer="word")), # type: ignore
        ("tfidf_char", TfidfVectorizer(analyzer="char")),
    ])),
    ("svd", TruncatedSVD()),
    ("clf", LinearSVC()),
])

# Init wandb

In [ ]:
wandbToken = "YOUR_WANDB_TOKEN"
wandb.login(key=wandbToken)
wandb.init(project="who-wrote-it-nlp", name="hyperparams-tuning")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/user/.netrc
wandb: Currently logged in as: qgurulev (zneus_speedating) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# Randomized search

In [9]:
ngram_ranges_word = [(1, 2), (1, 3), (2, 3)]
ngram_ranges_char = [(2, 4), (3, 5), (4, 6), (2, 5)]

rnd_params = {
    # word TF-IDF
    "features__tfidf_word__use_idf": [True, False],
    "features__tfidf_word__sublinear_tf": [True, False],
    "features__tfidf_word__norm": ["l1", "l2"],
    "features__tfidf_word__max_df": np.linspace(0.6, 0.9, 50).tolist(),
    "features__tfidf_word__min_df": np.linspace(0.001, 0.05, 50).tolist(),
    "features__tfidf_word__max_features": list(range(5000, 60001, 5000)),
    "features__tfidf_word__ngram_range": ngram_ranges_word,

    # char TF-IDF
    "features__tfidf_char__use_idf": [True, False],
    "features__tfidf_char__sublinear_tf": [True, False],
    "features__tfidf_char__norm": ["l1", "l2"],
    "features__tfidf_char__max_df": np.linspace(0.6, 0.9, 50).tolist(),
    "features__tfidf_char__min_df": np.linspace(0.001, 0.05, 50).tolist(),
    "features__tfidf_char__max_features": list(range(5000, 60001, 5000)),
    "features__tfidf_char__ngram_range": ngram_ranges_char,

    # SVD
    "svd__n_components": [10, 50, 100, 200, 300, 500, 600],

    # LinearSVC
    "clf__C": np.logspace(-2, 2, 50).tolist(),
}

In [12]:
search = RandomizedSearchCV(
    pipeline,
    rnd_params,
    n_iter=500,
    cv=StratifiedKFold(n_splits=4),
    scoring="f1_macro",
    n_jobs=4,
    verbose=2,
    random_state=1930912391,
)

In [13]:
search.fit(x_train, y_train) # type: ignore
for i, row in search.cv_results_.items():
    wandb.log({
        "type" : "random_search",
        "iteration": i,
        "mean_test_score": row['mean_test_score'],
        "std_test_score": row['std_test_score'],
        **{k: v for k, v in row["params"]}
    })

wandb.log({"best_score": search.best_score_, "best_params": search.best_params_})
model_path_rnd = joblib.dump(search.best_estimator_, os.path.join(MODEL_DIR, "rnd_model.joblib"))
if model_path_rnd is not None and os.path.exists(model_path_rnd.pop()):
    print(f"Model saved to {model_path_rnd[0]}")
    artifact = wandb.Artifact("best_model", type="model")
    artifact.add_file(model_path_rnd.pop())
    wandb.log_artifact(artifact)
    wandb.log_artifact(artifact, "random_search_best_model")
best_rnd = search.best_params_

Fitting 4 folds for each of 500 candidates, totalling 2000 fits
[CV] END clf__C=12.648552168552959, features__tfidf_char__max_df=0.6795918367346938, features__tfidf_char__max_features=55000, features__tfidf_char__min_df=0.05, features__tfidf_char__ngram_range=(2, 5), features__tfidf_char__norm=l1, features__tfidf_char__sublinear_tf=True, features__tfidf_char__use_idf=True, features__tfidf_word__max_df=0.6979591836734693, features__tfidf_word__max_features=55000, features__tfidf_word__min_df=0.024, features__tfidf_word__ngram_range=(1, 3), features__tfidf_word__norm=l2, features__tfidf_word__sublinear_tf=True, features__tfidf_word__use_idf=True, svd__n_components=500; total time=   0.0s
[CV] END clf__C=12.648552168552959, features__tfidf_char__max_df=0.6795918367346938, features__tfidf_char__max_features=55000, features__tfidf_char__min_df=0.05, features__tfidf_char__ngram_range=(2, 5), features__tfidf_char__norm=l1, features__tfidf_char__sublinear_tf=True, features__tfidf_char__use_idf

ValueError: 
All the 2000 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
2000 fits failed with the following error:
Traceback (most recent call last):
  File "/home/user/study/4course/who_wrote_it_nlp/venv/lib/python3.13/site-packages/pandas/core/indexes/base.py", line 3641, in get_loc
    return self._engine.get_loc(casted_key)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "pandas/_libs/index.pyx", line 168, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/index.pyx", line 176, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/index_class_helper.pxi", line 70, in pandas._libs.index.Int64Engine._check_type
KeyError: 'text'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/home/user/study/

# Grid search

In [ ]:
def make_range(value, *_args, pct=0.05, clip_min=None, clip_max=None, as_int=False):
    # 4 candidates around `value`: -10%, -5%, +5%, +10%
    factors = [1 - 2 * pct, 1 - pct, 1 + pct, 1 + 2 * pct]
    candidates = [value * f for f in factors]

    if clip_min is not None:
        candidates = [max(clip_min, c) for c in candidates]
    if clip_max is not None:
        candidates = [min(clip_max, c) for c in candidates]

    if as_int:
        candidates = [int(round(c)) for c in candidates]

    # remove duplicates while preserving order
    uniq = []
    for c in candidates:
        if c not in uniq:
            uniq.append(c)
    return uniq

grid_params = {
    # word TF-IDF
    "features__tfidf_word__use_idf":     [best_rnd["features__tfidf_word__use_idf"]],
    "features__tfidf_word__sublinear_tf":[best_rnd["features__tfidf_word__sublinear_tf"]],
    "features__tfidf_word__norm":        [best_rnd["features__tfidf_word__norm"]],
    "features__tfidf_word__ngram_range": [best_rnd["features__tfidf_word__ngram_range"]],
    "features__tfidf_word__max_df":      make_range(best_rnd["features__tfidf_word__max_df"]),
    "features__tfidf_word__min_df":      make_range(best_rnd["features__tfidf_word__min_df"]),
    "features__tfidf_word__max_features":make_range(best_rnd["features__tfidf_word__max_features"]),

    # char TF-IDF
    "features__tfidf_char__use_idf":     [best_rnd["features__tfidf_char__use_idf"]],
    "features__tfidf_char__sublinear_tf":[best_rnd["features__tfidf_char__sublinear_tf"]],
    "features__tfidf_char__norm":        [best_rnd["features__tfidf_char__norm"]],
    "features__tfidf_char__ngram_range": [best_rnd["features__tfidf_char__ngram_range"]],
    "features__tfidf_char__max_df":      make_range(best_rnd["features__tfidf_char__max_df"]),
    "features__tfidf_char__min_df":      make_range(best_rnd["features__tfidf_char__min_df"]),
    "features__tfidf_char__max_features":make_range(best_rnd["features__tfidf_char__max_features"]),

    # SVD
    "svd__n_components": [best_rnd["svd__n_components"]],

    # LinearSVC
    "clf__C": make_range(best_rnd["clf__C"], make_range(np.logspace(-2, 2, 50).tolist(), best_rnd["clf__C"]))
}

In [ ]:
grid_search = GridSearchCV(
    pipeline,
    grid_params,
    cv=StratifiedKFold(n_splits=4),
    scoring="f1_macro",
    n_jobs=4,
    verbose=2,
)
grid_search.fit(x_train, y_train) # type: ignore
for i, row in grid_search.cv_results_.items():
    wandb.log({
        "type" : "grid_search",
        "iteration": i,
        "mean_test_score": row['mean_test_score'],
        "std_test_score": row['std_test_score'],
        **{k: v for k, v in row["params"]}
    })
model_path_grid = joblib.dump(grid_search.best_estimator_, os.path.join(MODEL_DIR, "grid_model.joblib"))
if model_path_grid is not None and os.path.exists(model_path_grid.pop()):
    print(f"Model saved to {model_path_grid[0]}")
    artifact = wandb.Artifact("best_model_grid", type="model")
    artifact.add_file(model_path_grid.pop())
    wandb.log_artifact(artifact)
wandb.log({"grid_best_score": grid_search.best_score_, "grid_best_params": grid_search.best_params_})